In [9]:
"""
Validação completa do Data Lake S3 — forest-risk-datalake
Corre numa célula do Jupyter.
"""
import boto3
import pandas as pd
import io

s3 = boto3.client(
    's3',
    endpoint_url='http://localstack:4566',
    aws_access_key_id='test',
    aws_secret_access_key='test'
)

BUCKET = 'forest-risk-datalake'

def listar_parquets(prefixo):
    resp = s3.list_objects_v2(Bucket=BUCKET, Prefix=prefixo, MaxKeys=1000)
    return [o for o in resp.get('Contents', []) if o['Key'].endswith('.parquet')]

def tamanho_total(ficheiros):
    return sum(o['Size'] for o in ficheiros) / 1024

def ler_amostra(ficheiros):
    """
    Le ficheiros Parquet e reconstroi colunas de particao (ano, mes, grid_id)
    a partir do caminho — necessario porque o pandas partition_cols move
    essas colunas para o nome da pasta, nao para dentro do ficheiro.
    """
    dfs = []
    for f in ficheiros[:5]:
        try:
            obj = s3.get_object(Bucket=BUCKET, Key=f['Key'])
            df = pd.read_parquet(io.BytesIO(obj['Body'].read()))
            if len(df) > 0:
                # reconstroi colunas de particao a partir do caminho
                partes = f['Key'].split('/')
                for p in partes:
                    for col in ['ano', 'mes', 'grid_id']:
                        if p.startswith(f"{col}="):
                            df[col] = p.split('=')[1]
                dfs.append(df)
        except:
            continue
    if dfs:
        return pd.concat(dfs, ignore_index=True)
    return None

SEP  = "=" * 70
SEP2 = "-" * 70

print(SEP)
print("  VALIDAÇÃO DO DATA LAKE — forest-risk-datalake")
print(SEP)

resultados = {}

# ══════════════════════════════════════════════════════════════════════════════
# 1. HISTÓRICO NASA FIRMS
# ══════════════════════════════════════════════════════════════════════════════
historico = listar_parquets('hotspots/')
print(f"""
┌─ 1. HISTÓRICO NASA FIRMS ─────────────────────────────────────────────────┐
│  Origem  : CSV VIIRS (snpp + jpss1) processados pelo carga_historico_s3.py
│  Caminho : s3://forest-risk-datalake/hotspots/
│  Contém  : Hotspots de incêndio detectados por satélite em Portugal
└───────────────────────────────────────────────────────────────────────────┘""")

anos, zonas = set(), set()
for o in historico:
    for p in o['Key'].split('/'):
        if p.startswith('ano='):     anos.add(p.replace('ano=', ''))
        if p.startswith('grid_id='): zonas.add(p.replace('grid_id=', ''))

estado_h = '✅ OK' if len(historico) > 0 else '❌ VAZIO'
print(f"  Estado            : {estado_h}")
print(f"  Ficheiros Parquet : {len(historico)}")
print(f"  Tamanho total     : {tamanho_total(historico):.1f} KB")
print(f"  Anos cobertos     : {sorted(anos) if anos else 'nenhum'}")
print(f"  Zonas cobertas    : {len(zonas)} / 10 zonas de Portugal")

if historico:
    df_h = ler_amostra(historico)
    if df_h is not None:
        print(f"\n  Amostra de dados ({len(df_h)} linhas):")
        print(SEP2)
        cols_show = [c for c in ['acq_date','grid_id','frp','confidence','daynight','latitude','longitude'] if c in df_h.columns]
        print(df_h[cols_show].head(3).to_string(index=False))
        print(SEP2)
        print(f"\n  Colunas disponíveis:")
        for c in df_h.columns:
            descricoes = {
                'latitude':    'coordenada GPS',
                'longitude':   'coordenada GPS',
                'bright_ti4':  'temperatura de brilho canal 4 (K)',
                'bright_ti5':  'temperatura de brilho canal 5 (K)',
                'frp':         'Fire Radiative Power — intensidade do fogo (MW)',
                'acq_date':    'data de detecção pelo satélite',
                'acq_time':    'hora de detecção',
                'confidence':  'confiança da detecção (h=alta n=normal l=baixa)',
                'daynight':    'detecção de dia (D) ou noite (N)',
                'satellite':   'satélite que detectou (S-NPP ou NOAA-20)',
                'satelite':    'nome amigável do satélite',
                'ano':         'ano de detecção (partição)',
                'mes':         'mês de detecção (partição)',
                'dia':         'dia de detecção',
                'grid_id':     'zona de Portugal (adicionado pela carga)',
            }
            desc = descricoes.get(c, '')
            print(f"    {c:<20} {desc}")

resultados['historico_nasa'] = len(historico) > 0 and len(anos) >= 3


# ══════════════════════════════════════════════════════════════════════════════
# 2. METEOROLOGIA ERA5
# ══════════════════════════════════════════════════════════════════════════════
meteo = listar_parquets('meteorologia/')
print(f"""
┌─ 2. METEOROLOGIA ERA5 ────────────────────────────────────────────────────┐
│  Origem  : EDA_ERA5.py (dados Copernicus ERA5 reanalysis)
│  Caminho : s3://forest-risk-datalake/meteorologia/
│  Contém  : Temperatura, humidade, vento, precipitação históricos
└───────────────────────────────────────────────────────────────────────────┘""")

if meteo:
    anos_m = set()
    for o in meteo:
        for p in o['Key'].split('/'):
            if p.startswith('ano='): anos_m.add(p.replace('ano=', ''))
    print(f"  Estado            : ✅ OK")
    print(f"  Ficheiros Parquet : {len(meteo)}")
    print(f"  Tamanho total     : {tamanho_total(meteo):.1f} KB")
    print(f"  Anos cobertos     : {sorted(anos_m)}")
    df_m = ler_amostra(meteo)
    if df_m is not None:
        print(f"\n  Amostra de dados ({len(df_m)} linhas):")
        cols_show = [c for c in ['time','ano','mes','temp_c','rh','wind_speed_kmh','precip_mm_dia'] if c in df_m.columns]
        print(SEP2)
        print(df_m[cols_show].head(3).to_string(index=False))
        print(SEP2)
        print(f"\n  Colunas disponíveis:")
        for c in df_m.columns:
            descricoes = {
                'time':             'timestamp do registo ERA5',
                'latitude':         'coordenada GPS',
                'longitude':        'coordenada GPS',
                'ano':              'ano (partição)',
                'mes':              'mês (partição)',
                'dia':              'dia',
                'hora':             'hora UTC',
                'temp_c':           'temperatura do ar a 2m (°C)',
                'dewpoint_c':       'ponto de orvalho (°C)',
                'rh':               'humidade relativa (%)',
                'wind_speed_kmh':   'velocidade do vento a 10m (km/h)',
                'precip_mm_dia':    'precipitação total diária (mm)',
                'evap_mm_dia':      'evaporação diária (mm)',
                'risco_alto':       'flag: condições de alto risco de incêndio',
            }
            desc = descricoes.get(c, '')
            print(f"    {c:<25} {desc}")
else:
    print(f"  Estado            : ⏳ VAZIO")
    print(f"  Motivo            : EDA_ERA5.py ainda não correu (Pessoa B)")
    print(f"  Quando disponível : após EDA_ERA5.py gerar ERA5_Parquet/")

resultados['meteo_era5'] = len(meteo) > 0


# ══════════════════════════════════════════════════════════════════════════════
# 3. STREAMING SPARK — JOIN DOS 3 STREAMS
# ══════════════════════════════════════════════════════════════════════════════
streaming = listar_parquets('agregados_streaming/')
streaming = [o for o in streaming if '_spark_metadata' not in o['Key']]
print(f"""
┌─ 3. STREAMING SPARK — JOIN 3 STREAMS ─────────────────────────────────────┐
│  Origem  : spark_streaming_agregacao.py (job Spark Structured Streaming)
│  Caminho : s3://forest-risk-datalake/agregados_streaming/
│  Contém  : Médias e risco composto por zona, em janelas de 10 min
│  Fontes  :
│    sensor-events      → temperatura, humidade, vento (producer-sensores)
│    satellite-hotspots → hotspots NASA FIRMS em tempo real (producer-apis)
│    weather-data       → meteorologia IPMA (producer-apis)
└───────────────────────────────────────────────────────────────────────────┘""")

print(f"  Estado            : {'✅ OK' if streaming else '⏳ A aguardar janelas fecharem (~12 min)'}")
print(f"  Ficheiros Parquet : {len(streaming)}")
print(f"  Tamanho total     : {tamanho_total(streaming):.1f} KB")

if streaming:
    df_s = ler_amostra(streaming)
    if df_s is not None and len(df_s) > 0:
        # Detecta versao do job
        cols_join = ['n_hotspots', 'frp_medio', 'risco_composto']
        tem_join = all(c in df_s.columns for c in cols_join)
        versao = "✅ v2 — Join 3 streams" if tem_join else "⚠️  v1 — Só sensor-events (reinicia spark-streaming)"
        print(f"  Versão do job     : {versao}")

        print(f"\n  Amostra de dados ({len(df_s)} linhas):")
        print(SEP2)
        cols_show = [c for c in ['janela_inicio','grid_id','n_leituras_sensor',
                                  'n_hotspots','frp_medio','risco_composto'] if c in df_s.columns]
        print(df_s[cols_show].head(5).to_string(index=False))
        print(SEP2)

        print(f"\n  Colunas por origem:")
        grupos = {
            "sensor-events (producer-sensores)": [
                ('n_leituras_sensor', 'nº de leituras IoT na janela'),
                ('risk_medio_sensor', 'risco médio calculado pelo producer'),
                ('risk_maximo_sensor','risco máximo na janela'),
                ('temp_media',        'temperatura média (°C)'),
                ('humidade_media',    'humidade média (%)'),
                ('vento_medio',       'velocidade média do vento (km/h)'),
            ],
            "satellite-hotspots (producer-apis → NASA FIRMS)": [
                ('n_hotspots',        'nº de hotspots detectados por satélite'),
                ('frp_medio',         'Fire Radiative Power médio (MW)'),
                ('frp_maximo',        'Fire Radiative Power máximo (MW)'),
            ],
            "weather-data (producer-apis → IPMA)": [
                ('temp_max_media',    'temperatura máxima IPMA (°C)'),
                ('humidade_ipma',     'humidade média IPMA (%)'),
                ('vento_max_ipma',    'vento máximo IPMA (km/h)'),
                ('precipitacao_media','precipitação média (mm)'),
            ],
            "calculado pelo Spark (join dos 3)": [
                ('janela_inicio',     'início da janela temporal de 10 min'),
                ('janela_fim',        'fim da janela temporal de 10 min'),
                ('grid_id',           'zona geográfica de Portugal'),
                ('risco_composto',    '60% sensor + 25% FRP NASA + 15% vento IPMA → 0-100'),
            ],
        }
        for origem, cols in grupos.items():
            print(f"\n  [{origem}]")
            for col_nome, col_desc in cols:
                presente = '✅' if col_nome in df_s.columns else '⬜'
                print(f"    {presente} {col_nome:<25} {col_desc}")
    else:
        print(f"\n  Estrutura criada mas janelas ainda a fechar.")
        print(f"  Aguarda ~10-15 min e corre novamente.")
        if streaming:
            df_vazio = pd.read_parquet(io.BytesIO(
                s3.get_object(Bucket=BUCKET, Key=streaming[0]['Key'])['Body'].read()))
            print(f"  Colunas detectadas: {list(df_vazio.columns)}")
            cols_join = ['n_hotspots', 'frp_medio', 'risco_composto']
            tem_join = all(c in df_vazio.columns for c in cols_join)
            print(f"  Versão do job     : {'✅ v2 — Join 3 streams' if tem_join else '⚠️  v1 — reinicia spark-streaming'}")

resultados['streaming'] = len(streaming) > 0


# ══════════════════════════════════════════════════════════════════════════════
# RESUMO FINAL
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{SEP}")
print("  RESUMO FINAL")
print(SEP)
linhas = [
    ("Histórico NASA FIRMS", resultados['historico_nasa'],
     "corre: python /home/jovyan/work/carga_historico_s3.py"),
    ("Meteorologia ERA5",    resultados['meteo_era5'],
     "aguarda: EDA_ERA5.py (Pessoa B)"),
    ("Streaming Spark",      resultados['streaming'],
     "aguarda: ~12 min para janelas fecharem"),
]
for nome, ok, acao in linhas:
    if ok:
        print(f"  ✅  {nome:<30}")
    else:
        print(f"  ⏳  {nome:<30}  → {acao}")

n_ok = sum(resultados.values())
print(f"\n  {n_ok}/3 componentes prontos  ", end="")
if n_ok == 3:   print("✅  DATA LAKE COMPLETO")
elif n_ok >= 2: print("🟡  DATA LAKE PARCIAL")
else:           print("❌  DATA LAKE INCOMPLETO")
print(SEP)


  VALIDAÇÃO DO DATA LAKE — forest-risk-datalake

┌─ 1. HISTÓRICO NASA FIRMS ─────────────────────────────────────────────────┐
│  Origem  : CSV VIIRS (snpp + jpss1) processados pelo carga_historico_s3.py
│  Caminho : s3://forest-risk-datalake/hotspots/
│  Contém  : Hotspots de incêndio detectados por satélite em Portugal
└───────────────────────────────────────────────────────────────────────────┘
  Estado            : ✅ OK
  Ficheiros Parquet : 740
  Tamanho total     : 11009.6 KB
  Anos cobertos     : ['2020', '2021', '2022', '2023', '2024']
  Zonas cobertas    : 10 / 10 zonas de Portugal

  Amostra de dados (62 linhas):
----------------------------------------------------------------------
  acq_date        grid_id  frp confidence daynight  latitude  longitude
2020-01-08 PT-ALENTEJO-01 6.72          n        D  38.01133   -8.11061
2020-01-21 PT-ALENTEJO-01 3.87          n        D  38.13452   -8.07229
2020-01-21 PT-ALENTEJO-01 4.26          n        D  38.13630   -8.07214
----------